<a href="https://colab.research.google.com/github/MeouchyCharbel/Neural-Machine-Translation/blob/main/05_translation_interface.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -q gradio transformers sentencepiece sacremoses

In [2]:
from pathlib import Path

import torch
from google.colab import drive
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

drive.mount("/content/drive")

CHECKPOINT = Path("/content/drive/MyDrive/NMT_Finetuning/extended/checkpoint-1250")
BASE_MODEL = "Helsinki-NLP/opus-mt-en-fr"

if not CHECKPOINT.is_dir():
    raise FileNotFoundError(f"Fine-tuned checkpoint not found: {CHECKPOINT}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(str(CHECKPOINT)).to(device)
model.eval()

print("Model ready:", CHECKPOINT)
print("Device:", device)

Mounted at /content/drive


config.json:   0%|          | 0.00/1.42k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/778k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.34M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Model ready: /content/drive/MyDrive/NMT_Finetuning/extended/checkpoint-1250
Device: cuda


In [3]:
import gradio as gr


def translate(english_text):
    if not english_text or not english_text.strip():
        return "", None

    inputs = tokenizer(
        english_text.strip(),
        return_tensors="pt",
        truncation=True,
        max_length=64
    ).to(device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            num_beams=1,
            do_sample=False,
            max_new_tokens=64,
            return_dict_in_generate=True,
            output_scores=True
        )

        token_log_probs = model.compute_transition_scores(
            outputs.sequences,
            outputs.scores,
            normalize_logits=True
        )

        generated_ids = outputs.sequences[:, -token_log_probs.shape[1]:]
        valid_tokens = generated_ids.ne(tokenizer.pad_token_id)
        if tokenizer.eos_token_id is not None:
            valid_tokens = valid_tokens & generated_ids.ne(tokenizer.eos_token_id)

        translation = tokenizer.batch_decode(
            outputs.sequences,
            skip_special_tokens=True
        )[0]

        if valid_tokens.any().item():
            average_log_probability = token_log_probs[valid_tokens].mean()
            confidence_percent = round(
                torch.exp(average_log_probability).item() * 100, 1
            )
        else:
            confidence_percent = None

    return translation, confidence_percent


demo = gr.Interface(
    fn=translate,
    inputs=gr.Textbox(
        label="English sentence",
        placeholder="Type an English sentence here...",
        lines=2
    ),
    outputs=[
        gr.Textbox(label="French translation", lines=2),
        gr.Number(label="Average token confidence (%)", precision=1)
    ],
    title="English → French Translator",
    description=(
        "Extended fine-tuned Transformer (checkpoint-1250). "
        "The confidence figure summarizes generated-token probabilities; "
        "it is NOT the probability that the translation is correct."
    ),
    examples=[
        ["Hello, how are you?"],
        ["I am learning machine translation."],
        ["The weather is beautiful today."]
    ]
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://72c43866c018283339.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
